<a href="https://colab.research.google.com/github/fauzia-annisa/california_housing/blob/main/california_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
sns.set_style("whitegrid")

In [ ]:
cali = fetch_california_housing(as_frame=True)
cali_data = cali.frame
cali_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [ ]:
X = cali_data.drop('MedHouseVal', axis=1) # features
Y = cali_data['MedHouseVal'] # target = house price

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

model = LinearRegression() # The Algorithm
model.fit(X_train, Y_train)

preds = model.predict(X_test)
print("Model Error:", round(mean_squared_error(Y_test, preds), 2))

Model Error: 0.53


In [ ]:
map_ca = folium.Map(location=[36.7783, -119.4179], zoom_start=6)

for i in range(500):
    folium.CircleMarker(
            location=[cali_data.iloc[i]['Latitude'], cali_data.iloc[i]['Longitude']],
                    radius=3,
                            popup=f"${cali_data.iloc[i]['MedHouseVal']*100000:.0f}",
                                    color='#A8E6CF'
                                        ).add_to(map_ca)

map_ca

In [ ]:
# Combine test data with the model's predictions
results = X_test.copy()
results['actual'] = Y_test
results['predicted'] = preds
results['error'] = ( results['actual'] - results['predicted']).abs()

# Plot a random sample of 500 test houses
sample = results.sample(500, random_state=42)

error_map = folium.Map(location=[36.7783, -119.4179], zoom_start=6)

for _, row in sample.iterrows():
    if row['error'] < 0.3:
        color = 'green'    # model was close
    elif row['error'] < 0.7:
        color = 'orange'   # moderate error
    else:
        color = 'red'      # model was far off

    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4,
        color=color,
        fill=True,
        popup=f"Actual: ${row['actual']*100000:.0f}<br>Predicted: ${row['predicted']*100000:.0f}"
        ).add_to(error_map)
error_map

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, Y_train)
preds = model.predict(X_test)
print("Model Error:", round(mean_squared_error(Y_test, preds), 2))

Model Error: 0.26


In [ ]:
Xs = X_train.values
mu = Xs.mean(axis=0)
sd = Xs.std(axis=0)
Xn = np.c_[np.ones(len(Xs)), (Xs - mu) / sd]
y = Y_train.values
w = np.zeros(Xn.shape[1])
lr = 0.1
for step in range(1000):
    grad = (2 / len(y)) * Xn.T @ (Xn @ w - y)
    w = w - lr * grad
Xt = np.c_[np.ones(len(X_test)), (X_test.values - mu) / sd]
my_preds = Xt @ w
print("From-scratch error:", round(mean_squared_error(Y_test, my_preds), 2))

From-scratch error: 0.53


In [ ]:
cali_data.sample(5000, random_state=42).to_csv('cali.csv', index=False)
from google.colab import files
files.download('cali.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>